# World Cup Sentiment Tracker

## Data Exploration **(Real Tweets)** 

---

### First Dataset:

Source type:  
Replies/comments under official football tweets  
(less likely to be biased and/or noisy since official posts are usually neutral/promotional, but the replies are fan reactions)

Accounts:
- @FIFAWorldCup
- national team accounts
- major player/team accounts if relevant

Tweet types:
- goal announcements
- full-time result posts
- VAR/penalty/red-card posts
- lineup posts
- highlight clips

Collected unit:  
Individual reply/comment, not the official post itself

----
### Strategy:

1. Pick one match/event
2. Find official posts about that event
3. Pull replies under those posts
4. Store replies with metadata
5. Run baseline sentiment
6. Manually inspect model failures

---

## Real Football Tweet Collection

Goal:
Collect replies from official football event posts and evaluate how well the baseline sentiment model performs on real football fan language.

Questions:
1. Can we successfully collect real football discussion?
2. Does the baseline sentiment model perform reasonably well?
3. What types of football language cause model failures?

In [15]:
import sys
sys.path.append("../src")

from pathlib import Path

import pandas as pd

from api.getxapi_client import (
    fetch_all_replies,
    normalize_reply,
)
from data.match_linking import (
    link_replies_to_matches,
    load_match_schedule,
)
from data.preprocess import (
    annotate_replies,
    preprocessing_summary,
)
pd.set_option("display.max_colwidth", None)

SEED_TWEETS_PATH = Path("../data/reference/seed_tweets.csv")
MATCH_SCHEDULE_PATH = Path("../data/reference/matches_sample.csv")

RAW_OUTPUT_PATH = Path("../data/raw/real_replies_all.csv")
ANNOTATED_OUTPUT_PATH = Path("../data/processed/annotated_replies_all.csv")
ANALYSIS_OUTPUT_PATH = Path("../data/processed/analysis_replies_loose_all.csv")
MATCHLINKED_OUTPUT_PATH = Path("../data/processed/analysis_replies_matchlinked_all.csv")

# Keep this small during exploration to control API usage. Use None to fetch all pages.
MAX_PAGES_PER_TWEET = 1

In [16]:
seed_df = pd.read_csv(
    SEED_TWEETS_PATH,
    dtype={"parent_tweet_id": "string"},
    keep_default_na=False,
)

seed_df

,parent_tweet_id,source_account,parent_context,event,match,team,player,parent_tweet_timestamp,notes
0,2069367398214562161,FIFAWorldCup,unknown,unknown,multiple,,,,
1,2069359817282789383,FIFAWorldCup,unknown,unknown,multiple,,,,
2,2069390010445344865,FIFAWorldCup,unknown,unknown,multiple,,,,
3,2069469667668771053,FIFAWorldCup,unknown,unknown,multiple,,,,
4,2069502723125338295,FIFAWorldCup,unknown,unknown,multiple,,,,
5,2069547242428457247,FIFAWorldCup,unknown,unknown,multiple,,,,
6,2069633110703243324,FIFAWorldCup,unknown,unknown,multiple,,,,
7,2069887255305466045,FIFAWorldCup,unknown,unknown,multiple,,,,
8,2069888795743281188,FIFAWorldCup,unknown,unknown,multiple,,,,
9,2069890305675931724,FIFAWorldCup,unknown,unknown,multiple,,,,


In [17]:
def _seed_value(seed, column, default=None):
    value = seed.get(column, default)

    if value is None:
        return default

    if isinstance(value, str) and value.strip() == "":
        return default

    return value


def normalize_seed_replies(seed, raw_replies):
    rows = []

    for raw_reply in raw_replies:
        row = normalize_reply(
            raw=raw_reply,
            parent_tweet_id=str(seed["parent_tweet_id"]),
            match=_seed_value(seed, "match", "multiple"),
            event=_seed_value(seed, "event", "unknown"),
            source_account=_seed_value(seed, "source_account"),
            team=_seed_value(seed, "team"),
            player=_seed_value(seed, "player"),
        )
        row["parent_context"] = _seed_value(seed, "parent_context", "unknown")
        row["parent_tweet_timestamp"] = _seed_value(seed, "parent_tweet_timestamp")
        row["parent_notes"] = _seed_value(seed, "notes")
        rows.append(row)

    return rows

In [18]:
all_rows = []
collection_summary = []

for _, seed in seed_df.iterrows():
    parent_tweet_id = str(seed["parent_tweet_id"])
    raw_replies = fetch_all_replies(
        tweet_id=parent_tweet_id,
        max_pages=MAX_PAGES_PER_TWEET,
    )
    normalized_rows = normalize_seed_replies(seed, raw_replies)

    all_rows.extend(normalized_rows)
    collection_summary.append(
        {
            "parent_tweet_id": parent_tweet_id,
            "reply_rows": len(normalized_rows),
            "parent_context": _seed_value(seed, "parent_context", "unknown"),
            "event": _seed_value(seed, "event", "unknown"),
            "match": _seed_value(seed, "match", "multiple"),
            "team": _seed_value(seed, "team"),
            "player": _seed_value(seed, "player"),
        }
    )

collection_df = pd.DataFrame(collection_summary)
raw_df = pd.DataFrame(all_rows)

collection_df

,parent_tweet_id,reply_rows,parent_context,event,match,team,player
0,2069367398214562161,41,unknown,unknown,multiple,None,None
1,2069359817282789383,40,unknown,unknown,multiple,None,None
2,2069390010445344865,0,unknown,unknown,multiple,None,None
3,2069469667668771053,40,unknown,unknown,multiple,None,None
4,2069502723125338295,40,unknown,unknown,multiple,None,None
5,2069547242428457247,39,unknown,unknown,multiple,None,None
6,2069633110703243324,39,unknown,unknown,multiple,None,None
7,2069887255305466045,40,unknown,unknown,multiple,None,None
8,2069888795743281188,39,unknown,unknown,multiple,None,None
9,2069890305675931724,40,unknown,unknown,multiple,None,None


In [19]:
raw_df.shape

(396, 19)

In [20]:
raw_df.head()

,tweet_id,parent_tweet_id,url,timestamp,text,author_username,author_name,like_count,reply_count,view_count,match,team,player,event,source_account,source,parent_context,parent_tweet_timestamp,parent_notes
0,2069405786632253809,2069367398214562161,https://x.com/AkashSr27517223/status/2069405786632253809,Tue Jun 23 13:02:41 +0000 2026,@FIFAWorldCup https://t.co/IEnZyttQTH,AkashSr27517223,Akash,375,0,8709,multiple,None,None,unknown,FIFAWorldCup,getxapi,unknown,None,None
1,2069369674098225416,2069367398214562161,https://x.com/Cloudbet/status/2069369674098225416,Tue Jun 23 10:39:11 +0000 2026,@FIFAWorldCup 🇵🇹 Someone is begging Messi 🇦🇷 to stop scoring! ⚽\nhttps://t.co/Fu6iAL4Goo,Cloudbet,Cloudbet,196,10,11165,multiple,None,None,unknown,FIFAWorldCup,getxapi,unknown,None,None
2,2069378140057829866,2069367398214562161,https://x.com/asbunadja/status/2069378140057829866,Tue Jun 23 11:12:50 +0000 2026,@FIFAWorldCup Why u no include me? https://t.co/R6FfSCi9nc,asbunadja,asbunadja,545,5,15632,multiple,None,None,unknown,FIFAWorldCup,getxapi,unknown,None,None
3,2066596760558944382,2069367398214562161,https://x.com/gopuff/status/2066596760558944382,Mon Jun 15 19:00:37 +0000 2026,"Our delivery comes with a genius. Go learns what you like, builds your cart in seconds, and gets it all to your door in as fast as 15 minutes.",gopuff,Gopuff,2924,177,16431206,multiple,None,None,unknown,FIFAWorldCup,getxapi,unknown,None,None
4,2069378649095065643,2069367398214562161,https://x.com/LaudoVinny/status/2069378649095065643,Tue Jun 23 11:14:51 +0000 2026,@FIFAWorldCup @Arsenal_DJU what happens if it ends like this today? https://t.co/e4Mo0zQ2qX,LaudoVinny,LAU,338,10,17732,multiple,None,None,unknown,FIFAWorldCup,getxapi,unknown,None,None


In [21]:
raw_df.to_csv(
    RAW_OUTPUT_PATH,
    index=False,
)

RAW_OUTPUT_PATH

WindowsPath('../data/raw/real_replies_all.csv')

In [22]:
annotated_df = annotate_replies(raw_df)

schedule_df = load_match_schedule(MATCH_SCHEDULE_PATH)
annotated_df = link_replies_to_matches(
    annotated_df,
    schedule_df,
)

analysis_df = annotated_df[annotated_df["filter_reason"] == "keep"].reset_index(drop=True)

preprocessing_summary(annotated_df, analysis_df)

{'raw_rows': 396,
 'analysis_rows': 209,
 'removed_rows': 187,
 'filter_reasons': {'keep': 209,
  'low_relevance': 108,
  'unusable_text': 71,
  'spam': 8}}

In [23]:
analysis_df[
    [
        "parent_tweet_id",
        "text",
        "clean_text",
        "matched_entities",
        "inferred_teams",
        "inferred_players",
        "inferred_managers",
        "linked_match_id",
        "linked_match",
        "linked_match_confidence",
        "linked_match_method",
        "entity_confidence",
        "parent_context_confidence",
        "contextual_terms",
        "context_relevance_boost",
        "needs_context_review",
        "relevance_score",
        "filter_reason",
    ]
].head(20)

,parent_tweet_id,text,clean_text,matched_entities,inferred_teams,inferred_players,inferred_managers,linked_match_id,linked_match,linked_match_confidence,linked_match_method,entity_confidence,parent_context_confidence,contextual_terms,context_relevance_boost,needs_context_review,relevance_score,filter_reason
0,2069367398214562161,@FIFAWorldCup 🇵🇹 Someone is begging Messi 🇦🇷 to stop scoring! ⚽\nhttps://t.co/Fu6iAL4Goo,🇵🇹 Someone is begging Messi 🇦🇷 to stop scoring! ⚽,[messi],[Argentina],[Lionel Messi],[],2026_M041,Argentina vs Austria,high,team_entity_and_timestamp,1,0,[],0,False,2,keep
1,2069367398214562161,@FIFAWorldCup And there’s Ronaldo https://t.co/LyGTXyZH3d,And there’s Ronaldo,[ronaldo],[Portugal],[Cristiano Ronaldo],[],2026_M045,Portugal vs Uzbekistan,high,team_entity_and_timestamp,1,0,[],0,False,2,keep
2,2069367398214562161,@FIFAWorldCup Messi competing with these two in their prime is the most Messi thing ever lol,Messi competing with these two in their prime is the most Messi thing ever lol,[messi],[Argentina],[Lionel Messi],[],2026_M041,Argentina vs Austria,high,team_entity_and_timestamp,1,0,[],0,False,2,keep
3,2069367398214562161,@FIFAWorldCup @grok who is your favorite to win the golden boots of these three players?,who is your favorite to win the golden boots of these three players?,[win],[],[],[],NaN,NaN,none,no_team_context,1,0,[],0,False,3,keep
4,2069367398214562161,@FIFAWorldCup The goat is coming like a thief during knockouts 😈 https://t.co/1kO1320ow6,The goat is coming like a thief during knockouts 😈,[goat],[],[],[],NaN,NaN,none,no_team_context,1,0,[],0,False,3,keep
5,2069367398214562161,@FIFAWorldCup A 38 year old striker/playmaker is leading the pack.\n\nTruly Messi is the GOAT https://t.co/sf6pK1LnNF,A 38 year old striker/playmaker is leading the pack. Truly Messi is the GOAT,"[goat, messi, striker]",[Argentina],[Lionel Messi],[],2026_M041,Argentina vs Austria,high,team_entity_and_timestamp,3,0,[],0,False,8,keep
6,2069367398214562161,@FIFAWorldCup Messi standing there like the group dad 😂 https://t.co/O57oKxl3WL,Messi standing there like the group dad 😂,[messi],[Argentina],[Lionel Messi],[],2026_M041,Argentina vs Austria,high,team_entity_and_timestamp,1,0,[],0,False,2,keep
7,2069367398214562161,@FIFAWorldCup hey @grok add cristiano ronaldo to this trio poster,hey add cristiano ronaldo to this trio poster,"[cristiano, cristiano ronaldo, ronaldo]",[Portugal],[Cristiano Ronaldo],[],2026_M045,Portugal vs Uzbekistan,high,team_entity_and_timestamp,3,0,[],0,False,6,keep
8,2069367398214562161,@FIFAWorldCup Kane coming To table Tonight https://t.co/VJSnJQQpW8,Kane coming To table Tonight,[kane],[England],[Harry Kane],[],2026_M046,England vs Ghana,high,team_entity_and_timestamp,1,0,[],0,False,2,keep
9,2069367398214562161,@FIFAWorldCup @grok which player is missing,which player is missing,[player],[],[],[],NaN,NaN,none,no_team_context,1,0,[],0,False,3,keep


In [24]:
matchlinked_df = analysis_df[
    analysis_df["linked_match_id"].notna()
    & analysis_df["linked_match_confidence"].isin(["high", "medium"])
].reset_index(drop=True)

matchlinked_df.shape

(158, 38)

In [25]:
sample = analysis_df.sample(
    min(25, len(analysis_df)),
    random_state=42,
)

sample[
    [
        "parent_tweet_id",
        "text",
        "clean_text",
        "matched_entities",
        "inferred_teams",
        "inferred_players",
        "linked_match_id",
        "linked_match",
        "linked_match_confidence",
        "needs_context_review",
        "relevance_score",
    ]
]

,parent_tweet_id,text,clean_text,matched_entities,inferred_teams,inferred_players,linked_match_id,linked_match,linked_match_confidence,needs_context_review,relevance_score
30,2069359817282789383,@FIFAWorldCup Hope the only thing getting red cards today are my snack choices. May the best team finally win the group stage drama.,Hope the only thing getting red cards today are my snack choices. May the best team finally win the group stage drama.,"[team, win]",[],[],NaN,NaN,none,False,6
171,2069888795743281188,@FIFAWorldCup Classic win https://t.co/2Upx9tYxBo,Classic win,[win],[],[],NaN,NaN,none,False,3
84,2069547242428457247,@FIFAWorldCup Any Ghanaian here? Show the flag 🇬🇭🇬🇭🇬🇭🇬🇭🇬🇭,Any Ghanaian here? Show the flag 🇬🇭🇬🇭🇬🇭🇬🇭🇬🇭,[ghanaian],[Ghana],[],2026_M046,England vs Ghana,high,False,2
198,2069468618476200189,@FIFAWorldCup Messi's fans right now🤣🤣🤣 https://t.co/FcFd4StXrT,Messi's fans right now🤣🤣🤣,[messi],[Argentina],[Lionel Messi],2026_M041,Argentina vs Austria,high,False,2
60,2069502723125338295,@FIFAWorldCup What If\n\nAll 4 Were Playing In Same Team ⚽️ 🥅 🏟 🤔\n\nUndisputed Winners 🏆 https://t.co/fCDsI4RQ9X,What If All 4 Were Playing In Same Team ⚽️ 🥅 🏟 🤔 Undisputed Winners 🏆,[team],[],[],NaN,NaN,none,False,3
155,2069888795743281188,@FIFAWorldCup Team USA will be very tricky. They are really strong this year,Team USA will be very tricky. They are really strong this year,"[team, usa]",[United States],[],2026_M060,Turkey vs United States,high,False,5
45,2069469667668771053,@FIFAWorldCup Who is the GOAT 🐐 \nRonaldo or Messi https://t.co/c2FH7iHCsg,Who is the GOAT 🐐 Ronaldo or Messi,"[goat, messi, ronaldo]","[Argentina, Portugal]","[Cristiano Ronaldo, Lionel Messi]",2026_M045,Portugal vs Uzbekistan,medium,False,7
181,2069890305675931724,@FIFAWorldCup Congratulations 👏 team Canada,Congratulations 👏 team Canada,"[canada, team]",[Canada],[],2026_M050,Switzerland vs Canada,high,False,5
9,2069367398214562161,@FIFAWorldCup @grok which player is missing,which player is missing,[player],[],[],NaN,NaN,none,False,3
195,2069468618476200189,@FIFAWorldCup When Ronaldo scores\nThe whole world talks about Ronaldo \n\nWhen Messi scores\nThe whole world talk about Ronaldo \n\nThe Real GOAT.,When Ronaldo scores The whole world talks about Ronaldo When Messi scores The whole world talk about Ronaldo The Real GOAT.,"[goat, messi, ronaldo]","[Argentina, Portugal]","[Cristiano Ronaldo, Lionel Messi]",2026_M045,Portugal vs Uzbekistan,medium,False,7


In [26]:
annotated_df.to_csv(
    ANNOTATED_OUTPUT_PATH,
    index=False,
)

analysis_df.to_csv(
    ANALYSIS_OUTPUT_PATH,
    index=False,
)

matchlinked_df.to_csv(
    MATCHLINKED_OUTPUT_PATH,
    index=False,
)

{
    "annotated": str(ANNOTATED_OUTPUT_PATH),
    "analysis_loose": str(ANALYSIS_OUTPUT_PATH),
    "analysis_matchlinked": str(MATCHLINKED_OUTPUT_PATH),
}

{'annotated': '..\\data\\processed\\annotated_replies_all.csv',
 'analysis_loose': '..\\data\\processed\\analysis_replies_loose_all.csv',
 'analysis_matchlinked': '..\\data\\processed\\analysis_replies_matchlinked_all.csv'}

In [27]:
summary = {
    "seed_parent_tweets": len(seed_df),
    "raw_rows": len(raw_df),
    "annotated_rows": len(annotated_df),
    "analysis_rows": len(analysis_df),
    "matchlinked_rows": len(matchlinked_df),
    "linked_analysis_rows": int(analysis_df["linked_match_id"].notna().sum()),
    "ambiguous_analysis_rows": int((analysis_df["linked_match_confidence"] == "ambiguous").sum()),
    "retention_rate": len(analysis_df) / len(raw_df) if len(raw_df) else 0,
}

summary

{'seed_parent_tweets': 11,
 'raw_rows': 396,
 'annotated_rows': 396,
 'analysis_rows': 209,
 'matchlinked_rows': 158,
 'linked_analysis_rows': 158,
 'ambiguous_analysis_rows': 8,
 'retention_rate': 0.5277777777777778}

### Observations

Notebook 03 now collects replies from a seed list of parent tweets instead of one manually selected post. The parent IDs live in `data/reference/seed_tweets.csv`, alongside editable context fields for source account, event type, match, team, player, parent timestamp, and notes. Those context fields can be filled in later after manual parent-post inspection; the pipeline already uses them when present.

The working pipeline is now:

```text
Seed parent tweets
-> API reply collection
-> normalized raw reply rows
-> clean text
-> football entity detection
-> inferred team/player/manager metadata
-> schedule-based match linking
-> data quality filtering
-> loose and match-linked processed datasets
```

`annotated_df` preserves every collected row with audit columns. `analysis_df` keeps usable football-related rows for baseline sentiment exploration in Notebook 04. `matchlinked_df` is the stricter match-correlation subset: it keeps only analysis rows with a non-null `linked_match_id` and high or medium match-link confidence.

This keeps the project from overfitting Notebook 04 around one tweet thread. The next notebook should load the combined processed dataset and handle baseline sentiment evaluation, manual labeling, and failure analysis there. Actual fine-tuning should wait until there are enough manually labeled examples from different parent tweets and match contexts.
